### Transformar Datos "Employees" - String a JSON
1. Preprocesar la cadena JSON para corregir los problemas de calidad de los datos
2. Transformar cadena JSON en objeto JSON
3. Escribir los datos "transformados" en el schema "silver"

In [0]:
df_employees = spark.read.table("zenviro.bronze.employees_py")
display(df_employees)

#### 1. Preprocesar la cadena JSON para corregir los problemas de calidad de los datos

In [0]:
from pyspark.sql.functions import regexp_replace

df_fixed_value = (
    df_employees
    .select(
        regexp_replace("value", '"birth_date": (\\d{4}-\\d{2}-\\d{2})', '"birth_date": "$1"').alias("fixed_value")
    )
)
display(df_fixed_value)

#### 2. Transformar cadena JSON en objeto JSON

In [0]:
from pyspark.sql.functions import schema_of_json, col

get_schema = (
    df_fixed_value
    .select(
        schema_of_json(col("fixed_value")).alias("schema")
    )
)
display(get_schema.limit(1))

In [0]:
employees_schema = '''STRUCT<birth_date: STRING, department_id: BIGINT, email: STRING, employee_id: BIGINT, gender: STRING, hire_date: STRING, job_position_id: BIGINT, name: ARRAY<STRUCT<first_name: STRING, last_name: STRING>>, phone: STRING, status: BIGINT>'''

In [0]:
from pyspark.sql.functions import from_json

df_json_employees = (
    df_fixed_value
    .select(
        from_json("fixed_value", employees_schema).alias("json_value")
    )
)
display(df_json_employees)

#### 3. Escribir los datos "transformados" en el schema "silver"

In [0]:
df_json_employees.writeTo("zenviro.silver.employees_json_py").createOrReplace()

In [0]:
display(spark.table("zenviro.silver.employees_json_py"))